# 05 · Week 3 residual baseline (Kaggle)

Notebook này chỉ điều phối việc chạy `TinyResidualCNN_v1` và RGB-control.
Kết quả là in-domain validation; chưa phải unseen-generator evaluation.

In [ ]:
from pathlib import Path
import subprocess
import sys

# Nếu repo đã được attach, đặt REPO_DIR thủ công; nếu không notebook sẽ clone main.
REPO_DIR = Path('/kaggle/working/FaceTrace')
REPO_URL = 'https://github.com/AIVIETNAM-AIO-DinhBao/FaceTrace.git'
REPO_REF = 'main'
if not (REPO_DIR / 'src').is_dir():
    if REPO_DIR.exists() and any(REPO_DIR.iterdir()):
        REPO_DIR = Path('/kaggle/working/FaceTrace_repo')
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR))
print('Repo:', REPO_DIR)

In [ ]:
# Cài dependency bổ sung nếu Kaggle image chưa có. Không reinstall torch/CUDA.
%pip install -q 'PyYAML>=6' 'pandas>=2' 'Pillow>=10' 'scikit-learn>=1.4' 'tqdm>=4.66' 

In [ ]:
from pathlib import Path
import subprocess

# Auto-discover Who Is AI input; sửa DATA_ROOT thủ công nếu có nhiều candidate.
input_root = Path('/kaggle/input')
DATA_ROOT = None
if input_root.exists():
    candidates = sorted(p.parent.parent for p in input_root.rglob('manifest.csv')
                       if p.parent.name == 'train' and (p.parent / 'images').is_dir())
    if len(candidates) == 1:
        DATA_ROOT = candidates[0]
if DATA_ROOT is None:
    raise FileNotFoundError('Đặt DATA_ROOT tới thư mục chứa train/manifest.csv và train/images/.')
print('DATA_ROOT:', DATA_ROOT)

subprocess.run([
    sys.executable, str(REPO_DIR / 'scripts/create_splits.py'),
    '--manifest', str(DATA_ROOT / 'train' / 'manifest.csv'),
    '--train-images-root', str(DATA_ROOT / 'train'),
    '--image-path-base', str(REPO_DIR),
    '--output-dir', str(REPO_DIR / 'outputs/splits'),
], check=True)

In [ ]:
# CPU protocol tests trước khi train GPU.
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', str(REPO_DIR / 'tests'), '-v'], cwd=REPO_DIR, check=True)

In [ ]:
# Chạy seed 42 trước. Đổi configs/week03.yaml sang [42,43,44] sau khi seed 42 hoàn tất và kiểm tra artifact.
subprocess.run([
    sys.executable, str(REPO_DIR / 'scripts/train_residual.py'),
    '--config', str(REPO_DIR / 'configs/week03.yaml'),
], cwd=REPO_DIR, check=True)

In [ ]:
# Liệt kê artifact cần Save Version / download.
for path in sorted((REPO_DIR / 'outputs/week03').rglob('*')):
    if path.is_file():
        print(path.relative_to(REPO_DIR))